# Práctica 1: Ejercicios propuestos
# Inteligencia Artificial
# Grado en Ingeniería Informática - Ingeniería del Software
# Universidad de Sevilla

Los ejercicios que se plantean a continuación tienen como objetivo el practicar con la biblioteca [scikit-learn](https://scikit-learn.org) de Python.

### Ejercicio 1

El fichero `cars.csv` contiene información acerca de la idoneidad de una serie de coches, en función de los siguientes atributos discretos:

* Precio de compra (`buying`): posibles valores `vhigh`, `high`, `med`, `low`.
* Coste de mantenimiento (`maint`): posibles valores `vhigh`, `high`, `med`, `low`.
* Número de puertas (`doors`): posibles valores `2`, `3`, `4`, `5more`.
* Número de asientos (`persons`): posibles valores `2`, `4`, `more`.
* Tamaño del maletero (`lug_boot`): posibles valores `small`, `med`, `big`.
* Nivel de seguridad estimada (`safety`): posibles valores `low`, `med`, `high`.

La idoneidad de cada coche se indica mediante el atributo `acceptability`, que los clasifica como `unacc`, `acc`, `good` o `vgood`.

Se pide realizar lo siguiente:

1. Estimar mediante validación cruzada la tasa de acierto que obtendría un modelo naive Bayes para distintos valores del parámetro de suavizado.
2. Seleccionar el mejor valor de suavizado, entrenar un modelo naive Bayes a partir de todos los ejemplos usados para la validación cruzada y proporcionar su tasa de acierto sobre un conjunto de prueba reservado desde el principio.

In [25]:



# 1) Cargar datos
import pandas as pd
coches = pd.read_csv("cars.csv")

# 2) Separar X e y
X = coches.drop(columns=["acceptability"])
y = coches["acceptability"]

# 3) Split inicial (test reservado desde el principio)
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
   )

# 4) Preprocesado dentro de pipeline (evita fuga)
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OrdinalEncoder
from sklearn.naive_bayes import CategoricalNB
from sklearn.pipeline import Pipeline


cols_cat = X.columns.tolist()

preproceso = ColumnTransformer(
    transformers=[
        ("cat", OrdinalEncoder(), cols_cat)
    ],
    remainder="drop"
)

pipe = Pipeline(steps=[
        ("prep", preproceso),
        ("nb", CategoricalNB())
    ])

# 5) CV solo en train para elegir alpha
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold

param_grid = {
    "nb__alpha": [0.01, 0.1, 0.5, 1.0, 2.0, 5.0, 10.0]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="accuracy",
    cv=cv,
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

print("Mejor alpha:", grid.best_params_["nb__alpha"])
print("Accuracy media CV:", grid.best_score_)

# 6) Evaluación final en test reservado
from sklearn.metrics import accuracy_score
y_pred = grid.predict(X_test)
print("Accuracy test:", accuracy_score(y_test, y_pred))








Mejor alpha: 0.01
Accuracy media CV: 0.8552843614293936
Accuracy test: 0.8641618497109826


In [ ]:
# Conocer los parámetros que tiene cada clasificador
from sklearn.tree import DecisionTreeClassifier
from sklearn.naive_bayes import CategoricalNB
from sklearn.neighbors import KNeighborsClassifier

cart = DecisionTreeClassifier()
nb = CategoricalNB()
knn = KNeighborsClassifier()

print("CART:", sorted(cart.get_params().keys()))
print("NB:", sorted(nb.get_params().keys()))
print("kNN:", sorted(knn.get_params().keys()))


help(DecisionTreeClassifier)
help(CategoricalNB)
help(KNeighborsClassifier)

CART: ['ccp_alpha', 'class_weight', 'criterion', 'max_depth', 'max_features', 'max_leaf_nodes', 'min_impurity_decrease', 'min_samples_leaf', 'min_samples_split', 'min_weight_fraction_leaf', 'monotonic_cst', 'random_state', 'splitter']
NB: ['alpha', 'class_prior', 'fit_prior', 'force_alpha', 'min_categories']
kNN: ['algorithm', 'leaf_size', 'metric', 'metric_params', 'n_jobs', 'n_neighbors', 'p', 'weights']
Help on function accuracy_score in module sklearn.metrics._classification:

accuracy_score(y_true, y_pred, *, normalize=True, sample_weight=None)
    Accuracy classification score.

    In multilabel classification, this function computes subset accuracy:
    the set of labels predicted for a sample must *exactly* match the
    corresponding set of labels in y_true.

    Read more in the :ref:`User Guide <accuracy_score>`.

    Parameters
    ----------
    y_true : 1d array-like, or label indicator array / sparse matrix
        Ground truth (correct) labels. Sparse matrix is only su

### Ejercicio 2

Los púlsares son un tipo raro de estrella de neutrones que produce emisiones de radio detectables aquí en la Tierra. Son de considerable interés científico como sondas del espacio-tiempo, el medio interestelar y los estados de la materia.

A medida que los púlsares giran, su haz de emisión recorre el cielo y, cuando cruza nuestra línea de visión, produce un patrón detectable de emisión de radio de banda ancha. Como los púlsares giran rápidamente, este patrón se repite periódicamente. Por tanto, la búsqueda de púlsares implica buscar señales de radio periódicas con grandes radiotelescopios.

Cada púlsar produce un patrón de emisión algo diferente, que varía levemente con cada rotación. Por lo tanto, una detección de señal potencial conocida como «candidata» se promedia a lo largo de muchas rotaciones del púlsar, según lo determinado por la duración de una observación. A falta de información adicional, cada candidato podría describir un púlsar real. Sin embargo, en la práctica, casi todas las detecciones son causadas por interferencias de radiofrecuencia (RFI) y ruido, lo que dificulta encontrar señales legítimas.

El fichero `pulsar_stars.csv` contiene datos acerca de una serie de púlsares reales y de ejemplos espurios producidos por RFI y ruido. Cada candidato se describe mediante ocho atributos continuos extraídos de las señales recibidas.

Se pide realizar lo siguiente:

1. Dividir el conjunto de ejemplos en un subconjunto de entrenamiento (80&nbsp;% de los ejemplos) y un subconjunto de prueba (20&nbsp;% de los ejemplos). La división debe realizarse mediante muestreo estratificado, ya que la cantidad de ejemplos que se corresponden con púlsares reales es mucho menor que la de los que son interferencias y ruido.
2. Construir un árbol de decisión a partir del subconjunto de entrenamiento y calcular la matriz de confusión sobre el conjunto de prueba para cada combinación de los siguientes valores:
   - Máxima profundidad del árbol (argumento `max_depth`): de 1 a 5.
   - Cantidad mínima de ejemplos en las hojas (argumento `min_samples_leaf`): 1, 3 y 5.
   - Cantidad mínima de ejemplos para poder particionar (argumento `min_samples_split`): 10, 15 y 20.
3. De entre los árboles construidos en el apartado anterior seleccionar uno con máxima tasa de acierto sobre el conjunto de prueba, uno con máxima sensibilidad y uno con máxima precisión.

**Ayuda**: para los apartados 2 y 3 considerar el uso de [`ParameterGrid`](https://scikit-learn.org/stable/modules/generated/sklearn.model_selection.ParameterGrid.html) del módulo `model_selection`.

In [31]:
# 1. Cargar datos
import pandas as pd

pulsar_start = pd.read_csv('pulsar_stars.csv')
pulsar_start.head()

# 2. Separar X e y

X = pulsar_start.drop(columns='target_class')
y = pulsar_start['target_class']

# 3. dividir en conjunto de entrenamiento y pruebas
from sklearn.model_selection import train_test_split
(X_train, X_test, y_train, y_test) = train_test_split(X, y, random_state=42, test_size=.2, stratify=y)

# 4. Construimos una rejilla de parámetros 
from sklearn.model_selection import ParameterGrid
param_grid = {
    'max_depth': [1, 2, 3, 4, 5],
    'min_samples_leaf': [1, 3, 5],
    'min_samples_split': [10, 15, 20],
}

# 5. Para cada combinación de parámetros, construimos  el árbol
from sklearn.tree import DecisionTreeClassifier

resultados = []

for params in ParameterGrid(param_grid=param_grid):
    arbol = DecisionTreeClassifier(random_state=42, **params)
    arbol.fit(X_train, y_train)

    y_pred = arbol.predict(X_test)

    from sklearn.metrics import confusion_matrix, accuracy_score, precision_score, recall_score

    tn, fp, fn, tp = confusion_matrix(y_test, y_pred).ravel()

    resultados.append({
        "params": params,
        "accuracy": accuracy_score(y_test, y_pred),
        "precision": precision_score(y_test, y_pred, zero_division=0),
        "sensibilidad": recall_score(y_test, y_pred, zero_division=0),
        "matriz_confusion": [[tn, fp],[fn, tp]]
    })


mejor_accuracy = max(resultados, key=lambda r: r["accuracy"])
mejor_precision = max(resultados, key=lambda r: r["precision"])
mejor_sensibilidad = max(resultados, key=lambda r: r["sensibilidad"])

print("Mejor por accuracy:")
print(mejor_accuracy["params"])
print("Accuracy:", mejor_accuracy["accuracy"])
print("Precisión:", mejor_accuracy["precision"])
print("Sensibilidad:", mejor_accuracy["sensibilidad"])
print("Matriz de confusión:", mejor_accuracy["matriz_confusion"])
print()

print("Mejor por precisión:")
print(mejor_precision["params"])
print("Accuracy:", mejor_precision["accuracy"])
print("Precisión:", mejor_precision["precision"])
print("Sensibilidad:", mejor_precision["sensibilidad"])
print("Matriz de confusión:", mejor_precision["matriz_confusion"])
print()

print("Mejor por sensibilidad:")
print(mejor_sensibilidad["params"])
print("Accuracy:", mejor_sensibilidad["accuracy"])
print("Precisión:", mejor_sensibilidad["precision"])
print("Sensibilidad:", mejor_sensibilidad["sensibilidad"])
print("Matriz de confusión:", mejor_sensibilidad["matriz_confusion"])

Mejor por accuracy:
{'max_depth': 1, 'min_samples_leaf': 1, 'min_samples_split': 10}
Accuracy: 0.979608938547486
Precisión: 0.9322033898305084
Sensibilidad: 0.8384146341463414
Matriz de confusión: [[np.int64(3232), np.int64(20)], [np.int64(53), np.int64(275)]]

Mejor por precisión:
{'max_depth': 5, 'min_samples_leaf': 1, 'min_samples_split': 15}
Accuracy: 0.9793296089385475
Precisión: 0.9409722222222222
Sensibilidad: 0.8262195121951219
Matriz de confusión: [[np.int64(3235), np.int64(17)], [np.int64(57), np.int64(271)]]

Mejor por sensibilidad:
{'max_depth': 3, 'min_samples_leaf': 1, 'min_samples_split': 10}
Accuracy: 0.9776536312849162
Precisión: 0.8974358974358975
Sensibilidad: 0.8536585365853658
Matriz de confusión: [[np.int64(3220), np.int64(32)], [np.int64(48), np.int64(280)]]


### Ejercicio 3

El hormigón es el material más importante en la ingeniería civil. La resistencia a la compresión del hormigón es una función altamente no lineal de su edad y sus ingredientes.

El fichero `concrete_data.csv` contiene la siguiente información acerca de diferentes muestras de hormigón:

* Contenido de cemento (`Cement`).
* Contenido de escoria de alto horno (`Blast Furnace Slag`).
* Contenido de cenizas volantes (`Fly Ash`).
* Contenido de agua (`Water`).
* Contenido de superplastificantes (`Superplasticizer`).
* Contenido de agregados gruesos (`Coarse Aggregate`).
* Contenido de agregados finos (`Fine Aggregate`).
* Edad del hormigón (`Age`).

El objetivo es predecir la resistencia a la compresión (`Strength`) a partir de esos atributos continuos.

Se pide realizar lo siguiente:

1. Definir una tubería que concatene un transformador de columnas que normalice los atributos al intervalo $[0, 1]$ y un modelo $k$NN para regresión.
2. Realizar una búsqueda en rejilla para estimar mediante validación cruzada el coeficiente de determinación obtenido al aplicar la tubería a cada combinación de los valores 1 a 5 para el número de vecinos y las distancias manhattan y euclídea para la métrica.
3. Repetir los pasos 1 y 2 usando ahora la tipificación (es decir, restar la media y dividir por la desviación típica) como procedimiento de normalización de los atributos.
4. Seleccionar el mejor procedimiento de normalización, el mejor valor para el número de vecinos y la mejor métrica y entrenar un modelo $k$NN a partir de todos los ejemplos usados para la validación cruzada, proporcionando finalmente su coeficiente de determinación sobre un conjunto de prueba reservado desde el principio.

**Ayuda**: las clases [`MinMaxScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.MinMaxScaler.html) y [`StandardScaler`](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html) del módulo `preprocessing` implementan los procedimientos de normalización, mientras que la clase [`KNeighborsRegressor`](https://scikit-learn.org/stable/modules/generated/sklearn.neighbors.KNeighborsRegressor.html) del módulo `neighbors` implementa el modelo $k$NN para una tarea de regresión.

In [67]:
# 1. leer fichero
import pandas as pd
concrete_data = pd.read_csv('concrete_data.csv')

# 2. separar datos de atributos (X) y objetivo (y)
X = concrete_data.drop(columns='Strength')
y = concrete_data['Strength']

# 3 Reserva de datos para test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42, test_size=.2)

# Aptdo 1. 
# --- Normalizar datos al intervalo [0,1]
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler

normalizador = ColumnTransformer(
    transformers=[("normalizador", MinMaxScaler(), X.columns)],
    remainder="drop"
)

# --- tubería
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
tuberia_kNN = Pipeline([
    ('normalizador', normalizador),
    ('kNN_regresion', KNeighborsRegressor() )
])

# Aptdo 2.
from sklearn.model_selection import GridSearchCV

rejilla_de_parametros = {
    'kNN_regresion__n_neighbors': range(1,6),
    'kNN_regresion__metric':['manhattan', 'euclidean']
}

busqueda_en_rejilla = GridSearchCV(
    estimator=tuberia_kNN,
    scoring="r2",
    param_grid=rejilla_de_parametros,
    cv=10)

busqueda_en_rejilla.fit(X_train, y_train)

print(busqueda_en_rejilla.best_params_)
print(busqueda_en_rejilla.best_score_)

# Aptdo 3
# Repetir los pasos 1 y 2 usando ahora la tipificación (es decir, restar la media y dividir por la desviación típica) como procedimiento de normalización de los atributos.
# ---
from sklearn.preprocessing import StandardScaler

normalizador_2 = ColumnTransformer(
    transformers=[("normalizador_2", StandardScaler(), X.columns)],
    remainder="drop"
)

# --- tuberia
tuberia_kNN_2 = Pipeline([
    ('normalizador_2', normalizador_2),
    ('model', KNeighborsRegressor())
])

rejilla_de_parametros_2 = {
    'model__n_neighbors': range(1,6),
    'model__metric': ['manhattan', 'euclidean']
}

busqueda_en_rejilla_2 = GridSearchCV(
    estimator=tuberia_kNN_2,
    param_grid=rejilla_de_parametros_2,
    scoring="r2",
    cv=10
)

busqueda_en_rejilla_2.fit(X_train, y_train)
print(busqueda_en_rejilla_2.best_params_)
print(busqueda_en_rejilla_2.best_score_)

# Aptdo 4:  Seleccionar el mejor procedimiento de normalización, el mejor valor para el número de vecinos y la mejor métrica y entrenar un modelo $k$NN a partir de todos los ejemplos
# usados para la validación cruzada, proporcionando finalmente su coeficiente de determinación sobre un conjunto de prueba reservado desde el principio.

# --- elegir el mejor procedimiento de normalización
if busqueda_en_rejilla.best_score_ >= busqueda_en_rejilla_2.best_score_:
    mejor_busqueda = busqueda_en_rejilla
    nombre_normalizacion = "MinMaxScaler"
else:
    mejor_busqueda = busqueda_en_rejilla_2
    nombre_normalizacion = "StandardScaler"

# --- mejor modelo ya entrenado con todos los datos de entrenamiento
mejor_modelo = mejor_busqueda.best_estimator_

# --- Predicción sobre test y coeficiente de determinación final
from sklearn.metrics import r2_score
y_pred_test = mejor_modelo.predict(X_test)
r2_test = r2_score(y_test, y_pred_test)

# 4) Mostrar resultados finales del apartado 4
print("Mejor normalización:", nombre_normalizacion)
print("Mejores hiperparámetros:", mejor_busqueda.best_params_)
print("R2 medio en validación cruzada:", mejor_busqueda.best_score_)
print("R2 en test reservado:", r2_test)


{'kNN_regresion__metric': 'euclidean', 'kNN_regresion__n_neighbors': 5}
0.6720338878602321
{'model__metric': 'euclidean', 'model__n_neighbors': 4}
0.7017661921689637
Mejor normalización: StandardScaler
Mejores hiperparámetros: {'model__metric': 'euclidean', 'model__n_neighbors': 4}
R2 medio en validación cruzada: 0.7017661921689637
R2 en test reservado: 0.712081567493553


In [ ]:
from sklearn.neighbors import KNeighborsRegressor

help(KNeighborsRegressor)

Help on class KNeighborsRegressor in module sklearn.neighbors._regression:

class KNeighborsRegressor(sklearn.neighbors._base.KNeighborsMixin, sklearn.base.RegressorMixin, sklearn.neighbors._base.NeighborsBase)
 |  KNeighborsRegressor(n_neighbors=5, *, weights='uniform', algorithm='auto', leaf_size=30, p=2, metric='minkowski', metric_params=None, n_jobs=None)
 |
 |  Regression based on k-nearest neighbors.
 |
 |  The target is predicted by local interpolation of the targets
 |  associated of the nearest neighbors in the training set.
 |
 |  Read more in the :ref:`User Guide <regression>`.
 |
 |  .. versionadded:: 0.9
 |
 |  Parameters
 |  ----------
 |  n_neighbors : int, default=5
 |      Number of neighbors to use by default for :meth:`kneighbors` queries.
 |
 |  weights : {'uniform', 'distance'}, callable or None, default='uniform'
 |      Weight function used in prediction.  Possible values:
 |
 |      - 'uniform' : uniform weights.  All points in each neighborhood
 |        are we

### Ejercicio 4

Los [abulones](https://es.wikipedia.org/wiki/Haliotis) son una familia de moluscos gasterópodos. La edad de cada individuo está correlacionada con el número de anillos de su concha y, por tanto, puede determinarse cortando la concha a través del cono, tiñéndola y contando el número de anillos a través de un microscopio. Este procedimiento requiere mucho tiempo y es propenso a errores, por lo que sería preferible poder determinar la edad directamente a partir de medidas físicas más fáciles de obtener.

El fichero `abalone.csv` contiene la siguiente información de distintos individuos de abulones:

* Sexo (`Sex`): atributo discreto con posibles valores `M` (macho), `F` (hembra) e `I` (infante).
* Longitud (`Length`) en milímetros.
* Diámetro (`Diameter`) en milímetros.
* Altura (`Height`) en milímetros.
* Peso total (`Whole_weight`) en gramos.
* Peso sin la concha (`Shucked_weight`) en gramos.
* Peso intestinal (`Viscera_weight`) en gramos.
* Peso de la concha (`Shell_weight`) en gramos.

Se pide construir el mejor modelo posible para resolver la tarea de predecir el número de anillos (`Rings`) a partir de los atributos anteriores (entonces bastaría sumar 1.5 a ese número de anillos para obtener la edad, en años, del individuo).

In [108]:
# 1 leer datos
import pandas as pd
datos = pd.read_csv('abalone.csv')


# 2. Separar atributos y objetivo
X = datos.drop(columns='Rings')
y = datos['Rings']

# 3. Reservar datos para test
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=.2, random_state=42)

# 4. preprocesado
# Sex es nominal -> oneHotEnconder
# el resto de columnas son numéricas -> StandarScaler
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
preproceso = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(), ["Sex"]),
    ("num", StandardScaler(), [c for c in X.columns if c != "Sex"])
])

# 5. tubería
# --- el modelo se sustituye en la rejilla
from sklearn.pipeline import Pipeline
from sklearn.neighbors import KNeighborsRegressor
from sklearn.svm import SVR

pipe = Pipeline(steps=[
    ("prep", preproceso),
    ("model", KNeighborsRegressor())
])

# 6. rejilla de modelos más hiperparámetros
from sklearn.ensemble import RandomForestRegressor
param_grid =[
    {
        "model": [KNeighborsRegressor()],
        "model__n_neighbors": [3, 5, 7, 9, 11],
        "model__weights": ["uniform", "distance"],
        "model__metric": ['manhattan', 'euclidean']
    },
    {
        "model": [RandomForestRegressor(random_state=42)],
        "model__n_estimators": [200, 400],
        "model__max_depth": [None, 10, 20],
        "model__min_samples_split": [2, 5]

    },
    {
        "model": [SVR()],
        "model__kernel": ["rbf", "linear"],
        "model__C": [1, 10, 100],
        "model__epsilon": [0.1, 0.2]
    }

] 

# 7) Búsqueda en rejilla con validación cruzada
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(
    estimator=pipe,
    param_grid=param_grid,
    scoring="r2",
    cv=5,
    n_jobs=-1,
    refit=True
)

grid.fit(X_train, y_train)

# 8) Resultados CV
print("Mejor configuración:")
print(grid.best_params_)
print("Mejor R2 medio en validación cruzada:", grid.best_score_)

# 9) Evaluación final en test reservado
mejor_modelo = grid.best_estimator_
y_pred_test = mejor_modelo.predict(X_test)

from sklearn.metrics import r2_score, mean_absolute_error
print("R2 en test:", r2_score(y_test, y_pred_test))
print("MAE en test:", mean_absolute_error(y_test, y_pred_test))

Mejor configuración:
{'model': SVR(), 'model__C': 10, 'model__epsilon': 0.2, 'model__kernel': 'rbf'}
Mejor R2 medio en validación cruzada: 0.5591158888194798
R2 en test: 0.5656509059980777
MAE en test: 1.4856971043241955
